In [ ]:
import torch
from psutil import virtual_memory
import os
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn.utils
import gc

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    DataCollatorForLanguageModeling,
    get_linear_schedule_with_warmup
)


In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
    print('Not connected to a GPU')
else:
    print(gpu_info)

ram_gb = virtual_memory().total / 1e9
print(f'Your runtime has {ram_gb:.1f} GB RAM')
print('High-RAM runtime!' if ram_gb >= 20 else 'Low-RAM runtime')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEEDS = [42, 123, 2023, 777, 999]


In [ ]:

VISUALIZE_CLASS_DISTRIBUTION = True
USE_MIXED_PRECISION           = True
USE_LR_SCHEDULER              = True

In [ ]:
USE_MLM = True
MLM_EPOCHS = 15
CLS_EPOCHS = 100
PATIENCE = 10

learning_rates = [1e-4, 2e-4, 5e-4, 1e-5, 2e-5, 5e-5, 1e-6, 2e-6, 5e-6]




In [ ]:
MODEL_CASING = {
    "bert": "uncased",
    "scibert": "uncased",
    "biobert": "cased",
    "modernbert": "uncased",
    "pubmedbert_base": "uncased",
    "biolinkbert" : "uncased"
}


MODEL_MAP = {
    "bert": "bert-base-uncased",
    "scibert": "allenai/scibert_scivocab_uncased",
    "biobert": "dmis-lab/biobert-base-cased-v1.2",
    "modernbert": "answerdotai/ModernBERT-base",
    "pubmedbert_base": "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract",
    "biolinkbert_large": "michiyasunaga/BioLinkBERT-large",
    "biolinkbert_base": "michiyasunaga/BioLinkBERT-base"
}

selected_model = "biolinkbert"
BASE_MODEL = MODEL_MAP[selected_model]

In [ ]:
def clean_text(text):

    if not isinstance(text, str):
        return ""

    casing = MODEL_CASING.get(selected_model, "uncased")

    if casing == "uncased":
        text = text.lower()

    return re.sub(r'[^a-zA-Z0-9\s]', '', text)

In [ ]:
labeled = pd.read_csv("/content/drive/MyDrive/eq_5d/dataset/eq-5d-200-records.csv")
unlabeled = pd.read_csv("/content/drive/MyDrive/eq_5d/dataset/random_200_records.csv")

for df in [labeled, unlabeled]:
    for col in ['Title','Abstract']:
        df[col] = df[col].apply(clean_text)
    df["combined_text"] = df["Title"] + " [SEP] " + df["Abstract"]

labeled = sklearn.utils.shuffle(labeled, random_state=42)

print("Labeled data shape:", np.shape(labeled))
print("Columns:", labeled.columns)

In [ ]:
if VISUALIZE_CLASS_DISTRIBUTION:
    labeled["Label"].value_counts().plot(kind="bar", title="Class Distribution")
    plt.xlabel("Label")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

In [ ]:
def encode_classification(df):
    enc = tokenizer(
        df["combined_text"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    labels = torch.tensor(df["Label"].values)
    return TensorDataset(enc["input_ids"], enc["attention_mask"], labels)



In [ ]:
train_df, test_df = train_test_split(labeled, test_size=0.4, random_state=42)
_, val_df = train_test_split(test_df, test_size=0.3, random_state=42)

In [ ]:
train_ds = encode_classification(train_df)
val_ds = encode_classification(val_df)
test_ds = encode_classification(test_df)

In [ ]:
if USE_MLM:
    print("\nMLM PRETRAINING")
    mlm_texts = unlabeled["combined_text"].tolist()
    mlm_enc = tokenizer(
        mlm_texts,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    mlm_input_ids = mlm_enc["input_ids"].tolist()
    mlm_attention_mask = mlm_enc["attention_mask"].tolist()

    mlm_examples = [
        {"input_ids": ids, "attention_mask": mask}
        for ids, mask in zip(mlm_input_ids, mlm_attention_mask)
    ]

    mlm_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=0.15
    )

    mlm_loader = DataLoader(
        mlm_examples,
        batch_size=32,
        shuffle=True,
        collate_fn=mlm_collator
    )

    mlm_model = AutoModelForMaskedLM.from_pretrained(BASE_MODEL).to(device)
    mlm_optimizer = AdamW(mlm_model.parameters(), lr=5e-4)

    for epoch in range(MLM_EPOCHS):
        mlm_model.train()
        total_loss = 0

        for batch in tqdm(mlm_loader, desc=f"MLM Epoch {epoch+1}"):
            batch = {k: v.to(device) for k, v in batch.items()}
            mlm_optimizer.zero_grad()
            loss = mlm_model(**batch).loss
            loss.backward()
            mlm_optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} MLM Loss: {total_loss/len(mlm_loader):.4f}")

    torch.save(
        mlm_model.state_dict(),
        f"/content/drive/MyDrive/eq_5d/dataset/training/{selected_model}_without_keywords_mlm_pretrained_biolink_bert.pt"
    )
    print(f"Saved MLM model to /content/drive/MyDrive/eq_5d/dataset/training/{selected_model}_without_keywords_mlm_pretrained_biolink_bert.pt")

In [ ]:
all_preds = []
all_gold = None

for seed_idx, seed in enumerate(SEEDS):
    print(f"\nSEED {seed} ({seed_idx+1}/{len(SEEDS)})")
    set_seed(seed)

    best_f1 = 0
    best_lr = None

    for lr_idx, lr in enumerate(learning_rates):
        print(f"\n  LR {lr_idx+1}/{len(learning_rates)}: {lr}")

        # Clear cache
        torch.cuda.empty_cache()
        gc.collect()

        model = AutoModelForSequenceClassification.from_pretrained(
            BASE_MODEL,
            num_labels=labeled["Label"].nunique()
        )

        if USE_MLM:
            try:
                mlm_state_dict = torch.load(
                    f"/content/drive/MyDrive/eq_5d/dataset/training/{selected_model}_without_keywords_mlm_pretrained_biolink_bert.pt",
                    map_location=device
                )
                model.base_model.load_state_dict(mlm_state_dict, strict=False)
                print(f"    Loaded MLM weights into base model")
            except Exception as e:
                print(f"    Warning: Could not load MLM weights: {e}")

        model.to(device)
        optimizer = AdamW(model.parameters(), lr=lr)

        if USE_LR_SCHEDULER:
            total_steps = len(DataLoader(train_ds, batch_size=32)) * CLS_EPOCHS
            warmup_steps = int(0.1 * total_steps)
            scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=total_steps
            )

        scaler = torch.cuda.amp.GradScaler() if USE_MIXED_PRECISION else None

        early = 0
        best_epoch_f1 = 0

        for epoch in range(CLS_EPOCHS):
            model.train()
            epoch_loss = 0
            train_batches = 0

            for batch in DataLoader(train_ds, batch_size=32, shuffle=True):
                batch = [x.to(device) for x in batch]
                optimizer.zero_grad()

                if USE_MIXED_PRECISION:
                    with torch.cuda.amp.autocast():
                        loss = model(
                            input_ids=batch[0],
                            attention_mask=batch[1],
                            labels=batch[2]
                        ).loss
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss = model(
                        input_ids=batch[0],
                        attention_mask=batch[1],
                        labels=batch[2]
                    ).loss
                    loss.backward()
                    optimizer.step()

                if USE_LR_SCHEDULER:
                    scheduler.step()

                epoch_loss += loss.item()
                train_batches += 1

            model.eval()
            val_preds, val_gold = [], []
            for batch in DataLoader(val_ds, batch_size=32):
                batch = [x.to(device) for x in batch]
                with torch.no_grad():
                    logits = model(input_ids=batch[0], attention_mask=batch[1]).logits
                val_preds.extend(torch.argmax(logits, 1).cpu().numpy())
                val_gold.extend(batch[2].cpu().numpy())

            val_f1 = f1_score(val_gold, val_preds, average="micro")

            if (epoch + 1) % 10 == 0:
                print(f"    Epoch {epoch+1}: Loss={epoch_loss/train_batches:.4f}, Val F1={val_f1:.4f}")

            if val_f1 > best_f1:
                best_f1 = val_f1
                best_lr = lr
                best_epoch_f1 = epoch + 1
                torch.save(
                    model.state_dict(),
                    f"/content/drive/MyDrive/eq_5d/dataset/training/{selected_model}_without_keywords_seed{seed}_best.pt"
                )
                early = 0
            else:
                early += 1
                if early >= PATIENCE:
                    print(f"    Early stopping at epoch {epoch+1}")
                    break

        print(f"    Best F1 for LR {lr}: {best_f1:.4f} (epoch {best_epoch_f1})")

    print(f"\n  Best LR for seed {seed}: {best_lr} (Val F1={best_f1:.4f})")

    model.load_state_dict(torch.load(
        f"/content/drive/MyDrive/eq_5d/dataset/training/{selected_model}_without_keywords_seed{seed}_best.pt",
        map_location=device
    ))
    model.eval()

    preds, gold = [], []
    for batch in DataLoader(test_ds, batch_size=32):
        batch = [x.to(device) for x in batch]
        with torch.no_grad():
            logits = model(input_ids=batch[0], attention_mask=batch[1]).logits
        preds.extend(torch.argmax(logits, 1).cpu().numpy())
        gold.extend(batch[2].cpu().numpy())

    all_preds.append(np.array(preds))
    if all_gold is None:
        all_gold = np.array(gold)

    del model
    torch.cuda.empty_cache()
    gc.collect()

In [ ]:
f1_scores = [f1_score(all_gold, p, average="micro") for p in all_preds]
acc_scores = [accuracy_score(all_gold, p) for p in all_preds]

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print("\nF1 scores per seed:", f1_scores)
print("Mean F1 ± Std:", f"{np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print("\nAccuracy scores per seed:", acc_scores)
print("Mean Accuracy ± Std:", f"{np.mean(acc_scores):.4f} ± {np.std(acc_scores):.4f}")


In [ ]:
def bootstrap_ci_single(y_true, preds, metric, n=1000):
    """Bootstrap CI for a single prediction set"""
    scores = []
    N = len(y_true)
    for _ in range(n):
        idx = np.random.choice(N, N, replace=True)
        if metric == accuracy_score:
            scores.append(metric(y_true[idx], preds[idx]))
        else:
            scores.append(metric(y_true[idx], preds[idx], average="micro"))
    return np.percentile(scores, [2.5, 97.5])

best_idx = np.argmax(f1_scores)
best_preds = all_preds[best_idx]
best_seed = SEEDS[best_idx]

print(f"\nBest seed: {best_seed} (index {best_idx})")
print(f"95% CI F1 (best seed):", bootstrap_ci_single(all_gold, best_preds, f1_score))
print(f"95% CI Acc (best seed):", bootstrap_ci_single(all_gold, best_preds, accuracy_score))



In [ ]:
errors = []
for i in range(len(test_df)):
    if best_preds[i] != all_gold[i]:
        errors.append({
            "text": test_df.iloc[i]["combined_text"],
            "true": all_gold[i],
            "pred": best_preds[i]
        })

false_pos = [e for e in errors if e["pred"] == 1 and e["true"] == 0]
false_neg = [e for e in errors if e["pred"] == 0 and e["true"] == 1]

print(f"\nError Analysis (Seed {best_seed}):")
print(f"Total errors: {len(errors)} / {len(test_df)} ({len(errors)/len(test_df)*100:.1f}%)")
print(f"False Positives: {len(false_pos)}")
print(f"False Negatives: {len(false_neg)}")

if false_pos:
    print("\nSample False Positives (pred=1, true=0):")
    for i, e in enumerate(false_pos[:3]):
        print(f"  {i+1}. {e['text'][:200]}...")

if false_neg:
    print("\nSample False Negatives (pred=0, true=1):")
    for i, e in enumerate(false_neg[:3]):
        print(f"  {i+1}. {e['text'][:200]}...")

print("\n" + "="*60)
print("CLASSIFICATION REPORT (Best Seed)")
print("="*60)
print("\n", classification_report(all_gold, best_preds))

print("\nCONFUSION MATRIX:")
print(confusion_matrix(all_gold, best_preds))

results_df = pd.DataFrame({
    'Seed': SEEDS,
    'F1_Score': f1_scores,
    'Accuracy': acc_scores
})

results_summary = {
    'Model': selected_model,
    'Best_Seed': best_seed,
    'Mean_F1': np.mean(f1_scores),
    'Std_F1': np.std(f1_scores),
    'Mean_Accuracy': np.mean(acc_scores),
    'Std_Accuracy': np.std(acc_scores),
    'Total_Samples': len(labeled),
    'Train_Size': len(train_df),
    'Val_Size': len(val_df),
    'Test_Size': len(test_df)
}

results_df.to_csv(f"/content/drive/MyDrive/eq_5d/dataset/results/{selected_model}_without_keywords_results.csv", index=False)

print(f"\nResults saved to /content/drive/MyDrive/eq_5d/dataset/results/{selected_model}_without_keywords_results.csv")
print("\nSummary:")
for key, value in results_summary.items():
    print(f"  {key}: {value}")